---
title: "06. Observability & dashboard"
description: "Three non-overlapping observability layers plus a catalog/launcher dashboard over the results DB: what Part I delivers on Docker Compose, and what Azure operations add later."
---

## Outcome

The platform stays observable through three non-overlapping layers plus one
lightweight dashboard that unifies them: canonical run state, infra logs with
alerting, and operational dashboards. An operator can see what ran, what is
running, what failed and why, and can launch or rerun a workflow with an audit
trail.

The layers are identical in both parts of the course; their implementations
differ. **Part I** delivers canonical state fully: results-DB visibility through
the dashboard's catalog and results API, per-service logs via
`docker compose logs`, and the dashboard itself as a catalog + launcher over the
results DB and the local runner. Log Analytics/App Insights telemetry, Managed
Grafana, scheduled-query alert rules, and Entra-gated access arrive with the
Azure deployment in [chapter 13](13-azure-operations.ipynb). Nothing built here
needs re-plumbing when they land.


## Design — three layers, no duplication

| Layer | Answers | Part I (Compose) | Part II (Azure, ch. 13) |
|---|---|---|---|
| **Results DB** | Canonical run state: what ran / is running / failed and why | Postgres `results`, read through the dashboard catalog and `/api/results` | Unchanged: same DB, same readers |
| **Infra logs & alerting** | Is the machinery healthy? Did something silently stop? | `docker compose logs` per service | Log Analytics + App Insights, four scheduled-query alert rules |
| **Operational dashboards** | Trends across runs, deep dives into failures | Dashboard catalog + MLflow UI deep links | Azure Managed Grafana panels over logs and the results DB |

Each layer answers different questions from its own source and duplicates
nothing. Correlation travels on dimensions recorded once, in the results row:
the workflow `name`, the parent/child ids, and the model identity stored in each
run's output (`registered_version`, `mlflow_run_id`).

The **dashboard** is a catalog + launcher over this state, never a new source of
truth. It reads recent results rows, deep-links to MLflow (and Grafana once it
exists), and starts executions through an execution-plane endpoint: the local
runner in Part I, the ACA Jobs API under managed identity in Part II. On Azure,
human access splits into Entra groups (`ml-platform-operators` can launch,
`ml-platform-viewers` read-only); locally the boundary is the laptop itself.


## Build in `projects/ml-platform/`

```
projects/ml-platform/
├── src/dashboard/
│   ├── Dockerfile              # python:3.11-slim + FastAPI + psycopg
│   ├── requirements.txt        # fastapi/uvicorn/psycopg (+ azure SDKs for the Part II backend)
│   └── app.py                  # GET / catalog, GET /api/runs + /api/results,
│                               #   POST /api/runs/{job}/trigger, GET /healthz
├── demo/
│   ├── docker-compose.yml      # dashboard service: TRIGGER_BACKEND=local,
│   │                           #   RUNNER_URL=http://runner:8090, GRAFANA_URL=""
│   └── postgres/init/01-create-results.sql   # the results DB it reads
└── infra/modules/              # Part II counterparts, wired in chapter 13
    ├── observability/          # four Log Analytics scheduled-query alert rules
    └── dashboard/              # Dashboard ACA App: id-dashboard, Easy Auth, probes
```

The same `src/dashboard/app.py` serves both worlds; only its environment differs
([chapter 08](08-environment-contract.ipynb)). A mockup of the dashboard surface
lives at `projects/ml-platform/docs/mockups/workflow-dashboard.html`.


## How the pieces connect

### Dashboard app (`src/dashboard/app.py`)

The dashboard is **decoupled from execution**: it reads and triggers, stores
nothing authoritative. The whole surface:

| Route | Purpose |
|---|---|
| `GET /` | HTML catalog of the last 20 results rows, with deep links and local launch buttons |
| `GET /api/results`, `GET /api/runs` | Raw results rows for scripts or a richer frontend; `/api/results/{id}` fetches one |
| `GET /api/jobs` | The job catalog: names, accepted parameters, example payloads |
| `POST /api/runs/{job}/trigger` | Start an execution (train and batch get parameter-documented variants); returns the execution id immediately |
| `GET /api/executions/{id}` | Local-runner status plus the matching results row (`TRIGGER_BACKEND=local` only) |
| `GET /healthz` | Liveness probe |

Two trigger backends hide behind the one route. With `TRIGGER_BACKEND=local`
(Part I), the trigger posts `{"triggered_by", "parameters"}` to
`{RUNNER_URL}/api/jobs/{job}/run`; the runner validates parameters against a
per-job allowlist and runs the real entrypoints as subprocesses. With
`TRIGGER_BACKEND=aca` (Part II), the same route calls the ACA Jobs API
(`jobs.begin_start` under `id-dashboard`'s managed identity), passing the caller
identity as a `TRIGGERED_BY` env override so it lands in the results row
(chapter 13). Parameterized triggers work only against the local runner in this
POC; the ACA path starts the Job with its definition defaults.

Attribution rides on a single header contract: the caller's identity arrives as
`X-MS-CLIENT-PRINCIPAL-NAME` and is written to the results row's `triggered_by`.
On Azure, Entra Easy Auth injects the signed-in user's UPN automatically
(chapter 13): authorization by machine identity, attribution by human identity.
Locally the header is simulated; callers pass it by hand (`demo-user` in every
example), and the runner defaults an absent header to `demo-user`. The
application code cannot tell the difference, which is the point.

Deep links degrade gracefully. The Compose file sets `GRAFANA_URL=""`, so the
catalog renders a plain dash where the Grafana link would sit; `MLFLOW_UI_URL`
points at the published browser-reachable port (`http://localhost:15000`) rather
than the internal Compose DNS name, which a laptop browser cannot resolve.
Chapter 13 replaces both placeholders with real URLs once Managed Grafana and
public ingress exist.

Decoupling cuts both ways: the dashboard going down never stops jobs, and jobs
never require the dashboard. In-flight executions continue untouched, and the
same launch stays available by hand: locally, `curl` straight at the dashboard
API or the runner; on Azure, `az containerapp job start` with the same audit
trail.

### What Part II adds (`infra/modules/observability/`, `infra/modules/dashboard/`)

Chapter 13 layers the missing pieces on top of this foundation:

- **Log Analytics + App Insights** ingest container logs and telemetry from
  every Job and App.
- **Four scheduled-query alert rules** watch those signals:

| Alert | Condition | Severity |
|---|---|---|
| `job-failed` | Non-zero exit code in Job logs | 1 (critical) |
| `run-missed` | No Job execution in a two-hour window | 2 |
| `permanent-failures` | Permanent-failure messages exceed a threshold per hour | 2 |
| `batch-stalled` | Circuit-breaker message in logs | 1 (critical) |

  The action group is optional: left empty, rules exist but page nobody
  (dry-run mode). The module itself is always provisioned, with no `count` gate.
- **Azure Managed Grafana** builds operational panels over those logs plus the
  results DB.
- **The dashboard ACA App** re-runs this chapter's image with Easy Auth in front
  of the ingress, a read-only Postgres role from `grants.sql`, execution-start
  scoped to a dedicated `job_starter` role, and two-pass gating:
  `count = dashboard_image == "" || mlflow_image == "" ? 0 : 1`.

Locally the honest equivalents are modest: `docker compose logs <service>` for
logs, the catalog and `/api/results?status=FAILURE` for failure detection, and a
human noticing that an expected run never appeared.


## Golden-path position & acceptance evidence

This chapter builds the `batch / serve → dashboard` tail of the golden path.
Every upstream step already writes results rows, so visibility is mostly reading
what exists, plus launching what doesn't.

**Acceptance evidence — Part I (Compose):**

- The dashboard at `http://localhost:18000` lists recent runs read from the
  results DB and deep-links to the MLflow UI; the empty Grafana slot degrades to
  a dash instead of breaking the page.
- A forced failure appears within seconds in the catalog and in
  `GET /api/results?status=FAILURE`, with the error attached to the row.
- Launching training from the buttons or via curl records
  `triggered_by=demo-user` on the new results row, and polling the returned
  execution id tracks the run to `SUCCESS` or `FAILURE`.
- `docker compose logs runner` shows every manual run's subprocess output;
  `docker compose logs <service>` covers everything else.

**Acceptance evidence — Part II (added by chapter 13):**

- A forced job failure fires the `job-failed` alert; a stopped schedule is
  caught by `run-missed`.
- An operator launches a Job from the deployed dashboard and the run records
  their Entra identity in `triggered_by`; a viewer account cannot launch.
- Grafana panels chart run status and failure counts over time.


## Extensions (deferred from the production contract)

| Deferred | Contract | MVP substitute |
|---|---|---|
| Sampling strategy | `docs/06` | App Insights defaults once telemetry arrives (ch. 13) |
| Distributed tracing across planes | `docs/06` | Shared results-DB dimensions (`parent_id`, `name`, model version) |
| SLOs (99.5% API, 99% workflow) + runbook catalog | `docs/06` | Alert rules (ch. 13); failed runs visible in the catalog |
| Budget-monitoring alerts | `docs/06` | Cost review at tear-down |

Next: **[07 — LLM release artifacts](./07-llm-release-artifacts.ipynb)** ships an
LLM app through this exact same machinery.
